> **구분: 수업 후 복습·자습**

# 02. RNN과 LSTM — 25분

이 자료는 AI를 처음 배우는 비전공 학부생을 기준으로 작성했습니다. 수식과 용어가 한 번에 이해되지 않아도 괜찮습니다. 먼저 **입력 → 계산 → 출력 → 결과 해석** 의 흐름을 따라 실행하고, 수업 뒤 상세 설명을 다시 읽으세요.

본인 실습 컨테이너의 Python 커널에서 환경 셀부터 실행합니다. 설명 → 예제 → 파일 TODO → 학습자 검증 → 참고 정답 순서입니다.

실행 순서와 저장 위치는 [04.LLM README](README.md), 낯선 용어는 [04.LLM 용어 찾아보기](GLOSSARY.md)에서 바로 확인할 수 있습니다.

- **TODO는 JupyterLab 파일 편집기에서 `exercises/*.py`를 수정하고 저장합니다.** 노트북이 소스를 덮어쓰지 않습니다.
- 미완성 검증이 `AssertionError`로 실패하는 것은 정상입니다. 실패 메시지를 읽고 TODO를 고친 뒤 검증 셀을 다시 실행하세요.
- `load_op`는 저장된 파일을 다시 읽으므로 보통 커널을 재시작할 필요가 없습니다. 단, 일반 `import`로 읽은 모듈은 재시작 또는 reload가 필요할 수 있습니다.
- 완료한 앞 교시 TODO를 다음 훈련에서도 사용합니다. 참고 정답은 별도 `solutions/` 경로에 있으며, 강사가 안내하는 시점에 비교하고 자신의 코드로 재검증합니다.
- 표기 `[B,T,H]`는 텐서의 **shape** 입니다. `B`는 한 번에 처리하는 샘플 수(batch), `T`는 순서 길이(time/sequence), `H`는 내부 표현 크기(hidden), `V`는 token 종류 수(vocabulary)입니다.
- 컨테이너를 재생성하면 패키지가 포함된 최신 이미지가 필요하며, 소스와 `Data`는 호스트에 보존됩니다.

오류가 나면 가장 먼저 현재 노트북의 앞 셀을 순서대로 실행했는지, TODO 파일을 저장했는지, 오류가 난 텐서의 `shape`, `dtype`, `device`가 무엇인지 확인하세요.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 현재 실습의 입력·저장·실행 범위

열 파일은 `/workspace/04.LLM/02_rnn_lstm.ipynb`입니다. 첫 환경 셀은 `ROOT`, `COURSE`, `DATA`, `OUTPUTS`, `check`, `load_op`, `run_training`을 **이 커널에 다시 정의** 합니다. Python 경로와 PyTorch·CUDA 출력에서 CUDA가 `True`인지 확인하고 같은 Jetson의 다른 사용자와 GPU 실행 순서를 맞춥니다. 이 숫자 모델 훈련에는 Ollama 서버가 필요하지 않습니다.

입력은 `/workspace/Data/datasets/04.LLM/digits.json`과 `.sha256`입니다. 이 데이터 640개를 train 512개·validation 128개로 나누며, PAD/BOS/EOS=0/1/2, 숫자 0~9의 token ID=3~12입니다. 파일이 없으면 배포 데이터가 포함된 개인 자료 폴더인지 확인합니다. 이전 훈련 checkpoint 없이 새 훈련을 시작합니다.

`exercises/*.py`는 Jupyter 파일 편집기에서 수정·저장하세요. 노트북의 파일 출력 셀은 읽기 전용이고 `check(..., 'exercise')`는 저장한 최신 파일을 읽습니다. 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`·`loss.csv`, 자신의 가중치는 `/workspace/Data/checkpoints/users/04.LLM/`에 남습니다. `run_training`은 새 결과 폴더를 만들고 반환한 `report['checkpoint']`는 `Data` 기준 상대 경로입니다. 다른 노트북의 `report` 변수는 이어지지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

순서가 있는 입력을 읽을 때 현재 값만 보면 앞의 정보를 잃습니다. RNN은 hidden state로 과거를 전달하고 LSTM은 남길 기억과 지울 기억을 gate로 조절합니다. 이 자습의 목적은 같은 과제에서 상태 전달 방식의 차이를 확인하는 것입니다.

### 전체 과정에서 내 코드가 하는 일

```text
같은 입력·정답
  ↓
RNN/LSTM 상태 갱신 TODO
  ↓
훈련
  ↓
같은 검증 지표
  ↓
실패한 순서와 기억 거리 해석
```

현재 입력과 이전 상태가 새 상태에 어떻게 들어가는지 구현하고 shape를 확인합니다. 다른 데이터·출력 길이의 숫자를 같은 비교표에 섞지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 짧은 과제에서 RNN 점수가 더 높으면 LSTM이 불필요한가요?

**설명:** 그 결론은 이 실험 범위를 넘습니다. 과제 길이·훈련 조건·반복 변동을 확인해야 합니다. LSTM 구조가 기억을 조절한다는 원리와 이번 한 번의 점수는 구분해서 설명합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| hidden shape 오류·tuple 처리 오류 | RNN의 hidden과 LSTM의 (hidden, cell)을 구분하고 배치·시점·특징 축을 예제와 비교합니다. | 해당 check(..., exercise) 통과 |
| `ModuleNotFoundError: common` / 다른 함수가 로딩됨 | 작업 경로와 커널을 확인합니다. 다른 과목 common이 남으면 저장 후 해당 커널을 재시작하고 초기 셀부터 실행합니다. | 현재 04.LLM 모듈 import 성공 |
| 훈련 timeout·OOM·loss NaN | 직전 stdout/stderr와 설정을 보존합니다. 작은 quick profile부터 확인하고 중복 실행·입력 유한성·device를 점검합니다. | 작은 훈련 완료, 유한 loss와 metrics 저장 |

**예시에 없는 오류를 확인할 곳:** [LLM README](README.md), [contracts 검사](common/contracts.py), exercises/solutions의 해당 연산. 실행이 출력한 output 경로의 metrics.json과 터미널 오류를 확인하세요. [PyTorch 이슈](https://github.com/pytorch/pytorch/issues)는 버전과 최소 tensor 예제로 검색합니다.


[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 후 복습·자습**

본수업 훈련 120분은 [00_training_core.ipynb](00_training_core.ipynb)에서 진행합니다. 이 노트북 전체는 강사 선택 시연·자습이며 여기에 적힌 시간은 별도 자습 예상 시간입니다. 다섯 모델 구현 완료는 본수업 통과 조건이 아닙니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


> **구분: 수업 후 복습·자습**

## 시간·목표

반복 상태·기억 과제 6분 → LSTM gate 4분 → 두 모델 훈련 8분 → 결과 비교 7분. 선행: 01의 loss TODO.

### 자습 진행 경로

1. 환경과 1–2절에서 hidden state와 기억 과제를 확인합니다.
2. 5절의 LSTM gate 역할을 확인하고 6절에서 loss TODO를 재검증합니다.
3. RNN과 LSTM quick 훈련 셀을 실행합니다.
4. 7절 결과의 loss·token 정확도·sequence 정확도를 비교합니다.
5. 3–4절 shape·gradient 상세 설명과 8절 오류 표는 수업 후 복습합니다.

각 셀의 **구분** 표시를 확인하세요. `수업 후 복습·자습` 셀은 건너뛰어도 다음 수업 핵심 셀을 실행할 수 있습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
from pathlib import Path
import importlib, json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.contracts import check
from common.loading import load_op
import torch
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '훈련은 Jetson GPU에서 수행합니다. 강사에게 환경 점검을 요청하세요.'
def run_training(kind, variant='exercise'):
    before = {p.resolve() for p in OUTPUTS.glob('*/metrics.json')}
    command = [sys.executable, str(COURSE / 'scripts/train.py'), '--model', kind, '--variant', variant]
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    created = [p for p in OUTPUTS.glob('*/metrics.json') if p.resolve() not in before]
    assert len(created) == 1, f'새 metrics.json이 정확히 1개여야 합니다: {created}'
    metrics_path = created[0]
    report = json.loads(metrics_path.read_text())
    assert report['kind'] == kind and report['variant'] == variant and report['profile'] == 'quick'
    report['_metrics_path'] = str(metrics_path.relative_to(DATA))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    if kind in ('rnn','lstm'):
        print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    else:
        print('자유 생성 token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('자유 생성 시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    print('실행 경로:', report['run_path'])
    return report


> **구분: 수업 후 복습·자습**

## 1. 순서가 있는 데이터와 hidden state

<!-- procedure:lesson-12-8aa2ed:start -->
![시점마다 기억을 전달하기](../교재그림/절차/lesson-12.svg)

LSTM은 hidden 외에 cell state도 전달합니다. 시퀀스 진행과 optimizer 업데이트는 다른 반복입니다.
<!-- procedure:lesson-12-8aa2ed:end -->


사진처럼 한 번에 볼 수 있는 데이터와 달리 문장, 센서 로그, 소리에는 **순서** 가 있습니다. `오늘 비가 안 온다`에서 `안`의 위치가 바뀌면 의미가 달라지고, 온도 센서에서도 과거 값의 흐름이 현재 상태 판단에 영향을 줍니다.

**RNN(Recurrent Neural Network, 순환 신경망)** 은 시퀀스를 왼쪽에서 오른쪽으로 한 위치씩 읽습니다. 각 시점 `t`에서 현재 입력 `x_t`와 직전의 **hidden state** `h_(t-1)`를 결합해 새 상태를 만듭니다.

`h_t = tanh(W_x x_t + W_h h_(t-1) + b)`

- `x_t`: 현재 위치의 embedding `[B,E]`
- `h_(t-1)`: 이전까지 읽은 정보를 요약한 상태 `[B,H]`
- `W_x`, `W_h`, `b`: 훈련으로 배우는 파라미터
- `tanh`: 값을 -1–1 범위로 누르는 비선형 함수

같은 RNN 셀과 같은 파라미터를 모든 시점에 반복 사용합니다. 그림에서 셀이 여러 개로 펼쳐져 보여도 시점마다 독립된 모델을 만드는 것이 아닙니다. hidden state는 과거 원문 전체를 그대로 저장하는 메모리라기보다, 과제에 도움이 되도록 압축된 실수 벡터입니다.


> **구분: 수업 후 복습·자습**

## 2. 이번 기억 과제

이번 과제는 각 위치에서 **한 시점 전 입력** 을 출력하는 것입니다.

```text
입력 위치       0  1  2  3  4
입력 token      3  8  5  9  4
목표 token    BOS  3  8  5  9
```

0번 위치에는 이전 입력이 없으므로 BOS를 정답으로 둡니다. 1번 위치에서 입력 8을 보고 출력 3을 내야 하므로, 현재 token만 선형 변환해서 복사하는 모델로는 풀 수 없습니다. 직전 정보가 hidden state를 통해 전달되어야 합니다.

이 과제는 RNN의 상태 전달을 눈으로 확인하기 위한 **sequence tagging** 실험입니다. 자연어 LLM의 일반적인 next-token prediction은 앞 문맥으로 다음 token을 예측하므로 입력과 목표 정렬이 다릅니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
from common.training import load_data
(train_pair, val_pair), _ = load_data('rnn', 'cpu')
print('입력 shape:', train_pair[0].shape, '정답 shape:', train_pair[1].shape)
print('입력:', train_pair[0][0].tolist(), '정답:', train_pair[1][0].tolist())
from common.models import RecurrentTagger
print(RecurrentTagger('rnn'))
print(RecurrentTagger('lstm'))

> **구분: 수업 후 복습·자습**

## 3. RNN 안에서 shape 추적

이 자료의 `RecurrentTagger`는 다음 경로를 사용합니다.

```text
source [B,T] --Embedding(V,32)--> vectors [B,T,32]
             --RNN(32,64)------> states  [B,T,64]
             --Linear(64,V)----> logits  [B,T,13]
             --argmax(V축)-----> tokens  [B,T]
```

`batch_first=True`이므로 RNN 입력과 출력의 첫 축이 `B`입니다. PyTorch 모듈에 따라 기본이 `[T,B,H]`인 경우도 있으므로 문서와 설정을 확인해야 합니다. 마지막 hidden state의 shape는 층 수와 양방향 여부까지 포함해 `[num_layers × directions, B, H]`입니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
demo_source = train_pair[0][:2]
demo_model = RecurrentTagger('rnn')
demo_vectors = demo_model.embedding(demo_source)
demo_states, demo_last_hidden = demo_model.recurrent(demo_vectors)
demo_logits = demo_model.output(demo_states)
print('source      :', demo_source.shape)
print('embedding   :', demo_vectors.shape)
print('all states :', demo_states.shape)
print('last hidden:', demo_last_hidden.shape)
print('logits      :', demo_logits.shape)


> **구분: 수업 후 복습·자습**

## 4. 긴 기억이 어려운 이유: gradient 소실과 폭주

RNN은 여러 시점에서 같은 변환을 반복합니다. 역전파도 마지막에서 앞쪽으로 여러 번 곱셈을 거칩니다. 이 값들이 반복해서 1보다 작게 작용하면 앞 시점의 gradient가 거의 0이 되는 **vanishing gradient(기울기 소실)** 가 생기고, 1보다 크게 작용하면 매우 커지는 **exploding gradient(기울기 폭주)** 가 생길 수 있습니다.

기울기가 소실되면 오래전 정보와 관련된 파라미터가 거의 학습되지 않습니다. 폭주하면 loss가 불안정해지거나 `nan`이 될 수 있습니다. 이 실습의 훈련 코드는 gradient norm을 1.0으로 제한하는 **gradient clipping** 으로 폭주를 완화합니다. clipping은 긴 기억 문제를 전부 해결하는 방법은 아닙니다.


> **구분: 수업 후 복습·자습**

## 5. LSTM의 cell state와 gate

**LSTM(Long Short-Term Memory)** 은 RNN의 hidden state 외에 비교적 직접적인 정보 통로인 **cell state** `c_t`를 둡니다. 그리고 0–1 값을 내는 sigmoid gate로 정보를 조절합니다.

- **forget gate** : 이전 cell state에서 얼마나 유지할지 정함
- **input gate** : 현재 후보 정보를 얼마나 기록할지 정함
- **output gate** : cell state 중 얼마를 현재 hidden state로 보여줄지 정함

gate는 사람이 `이 정보는 중요하다`고 규칙을 작성하는 장치가 아닙니다. gate를 계산하는 파라미터도 데이터로 훈련됩니다. LSTM은 긴 의존성을 배우기 쉽게 설계되었지만 무한한 기억을 보장하지 않고, 모든 짧은 과제에서 단순 RNN보다 반드시 높은 정확도를 내지도 않습니다. 모델 크기, 데이터, 초기값, 학습률과 훈련 step도 결과에 영향을 줍니다.

PyTorch LSTM의 마지막 상태는 `(h_n, c_n)` 두 텐서입니다. 둘의 shape는 같을 수 있지만 역할은 다릅니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
# 3절 shape 예제를 건너뛴 자습 경로에서도 입력을 준비합니다.
demo_source = train_pair[0][:2]
lstm_demo = RecurrentTagger('lstm')
lstm_states, (h_n, c_n) = lstm_demo.recurrent(lstm_demo.embedding(demo_source))
print('모든 시점의 hidden:', lstm_states.shape)
print('마지막 hidden h_n :', h_n.shape)
print('마지막 cell c_n   :', c_n.shape)


> **구분: 수업 후 복습·자습**

## 6. 훈련 전 확인

`exercises/loss.py`의 loss 구현이 그대로 두 모델 훈련에 사용됩니다. 잘못된 loss로 훈련 시간을 소비하지 않도록 검증을 먼저 실행합니다.
훈련은 quick 설정으로 실행하고 각 모델의 결과 폴더를 자동으로 새로 만듭니다.

### 앞선 loss TODO의 위치와 현재 필요한 내용

원래 설명은 [00_training_core.ipynb의 「수업 핵심 TODO — loss」](00_training_core.ipynb)와 [01_training_basics.ipynb의 「자습 구현 TODO — loss」](01_training_basics.ipynb)에 있습니다. 두 노트북이 편집하는 실제 파일은 **[exercises/loss.py](exercises/loss.py)** 하나입니다. 이전 커널을 열 필요 없이 이 파일의 `token_loss(logits, targets)`를 아래 계약으로 완성·저장하세요.

- 입력 logits는 raw 실수 점수 `[B,T,V]`, targets는 같은 장치의 정수 class 번호 `[B,T]`(값 `0..V-1`)입니다.
- 마지막 후보 축 V를 남겨 logits를 `[B*T,V]`로, targets를 같은 순서로 `[B*T]`로 펼칩니다. `reshape(-1, logits.shape[-1])`와 `reshape(-1)`의 역할을 구분합니다.
- `F.cross_entropy`에 두 값을 전달하고 평균 loss인 0차원 텐서를 반환합니다. softmax를 미리 적용하거나 `.item()`, `.detach()`로 gradient 연결을 끊지 않습니다.
- 모든 점수가 같은 후보 4개라면 loss는 약 `1.3863`이지만 이 값을 고정 반환하면 안 됩니다. 실제 검사에서는 정답 점수를 낮출 때 loss 증가와 backward의 비영 기울기까지 확인합니다.
- 이 노트북의 아래 `check('loss', 'exercise')`가 저장한 파일을 다시 읽습니다. 통과 후 훈련하세요. 참고 정답은 [solutions/loss.py](solutions/loss.py)에 있으며 이를 실행하는 것만으로 exercise 파일은 완성되지 않습니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
check('loss', 'exercise')
print('learner_verification=PASS')

In [ ]:
# 구분: 수업 후 복습·자습 셀
rnn_report = run_training('rnn')

In [ ]:
# 구분: 수업 후 복습·자습 셀
lstm_report = run_training('lstm')

> **구분: 수업 후 복습·자습**

## 7. 측정 결과를 해석하기

훈련 스크립트는 파라미터가 무작위인 **훈련 전(before)** 과 지정된 step을 마친 **훈련 후(after)** 를 같은 검증 데이터에서 측정합니다.

- **validation loss** : 정답 token에 얼마나 높은 점수를 주는지를 요약합니다. 낮아지는 방향이 좋습니다.
- **token accuracy** : 모든 샘플의 모든 위치 중 정답과 같은 위치 비율입니다. 예를 들어 100개 위치 중 93개가 맞으면 0.93입니다.
- **sequence accuracy** : 시퀀스의 모든 위치가 맞은 샘플 비율입니다. 한 위치라도 틀리면 그 시퀀스는 실패이므로 token accuracy보다 엄격합니다.
- **parameters** : 훈련 가능한 숫자의 총개수입니다. 많다고 자동으로 좋은 모델이 되는 것은 아니며 메모리·계산 비용도 늘어납니다.
- **training_seconds** : 이 장비에서 훈련 루프에 걸린 시간입니다. 첫 CUDA 초기화, 평가, 저장 시간을 모두 포함한 수업 체감 시간과는 다릅니다.

정확도가 높아졌다면 모델이 이 작은 규칙을 학습했다는 증거입니다. 이것만으로 길이가 더 긴 시퀀스, 새로운 token 종류, 자연어에서도 잘 작동한다고 결론낼 수는 없습니다. 그런 조건은 데이터 분포 밖의 일반화 문제이므로 별도 검증해야 합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
for report in (rnn_report, lstm_report):
    print('\n모델:', report['kind'])
    print('파라미터:', report['parameters'], '훈련 초:', round(report['training_seconds'],2))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
    print('sequence 정확도:', report['before']['sequence_accuracy'], '→', report['after']['sequence_accuracy'])
    print('예시:', report['after']['examples'])

> **구분: 수업 후 복습·자습**

## 8. 자주 생기는 오해와 오류

- **"hidden state가 입력을 그대로 저장한다"** : hidden state는 과제를 풀기 좋게 학습된 압축 표현입니다. 어떤 차원이 특정 token을 뜻한다고 미리 정해져 있지 않습니다.
- **"LSTM 정확도가 같으니 gate가 쓸모없다"** : 현재 과제는 길이 5의 매우 짧은 기억 과제입니다. 구조 차이를 평가하려면 긴 의존성, 여러 seed와 적절한 하이퍼파라미터가 필요합니다.
- **"훈련 정확도가 높으니 검증도 볼 필요 없다"** : 모델은 훈련 예제를 외울 수 있습니다. 사용하지 않은 검증 입력의 성능을 따로 봐야 합니다.
- **shape가 `[T,B,H]`처럼 뒤집힘** : `batch_first=True` 설정과 입력 shape를 확인합니다.
- **앞 노트북에서는 통과했는데 훈련이 실패함** : [exercises/loss.py](exercises/loss.py)를 저장했는지 확인하고 검증 셀을 다시 실행합니다.
- **같은 실행 이름 폴더가 이미 있다는 오류** : 결과를 덮어써 출처를 잃지 않도록 의도적으로 막은 것입니다. 자동 이름을 쓰거나 새로운 `--run-name`을 지정합니다.
- **첫 실행만 느림** : CUDA context 초기화와 라이브러리 준비가 포함될 수 있습니다. 보고서의 훈련 루프 시간과 전체 셀 시간을 구분합니다.


> **구분: 수업 후 복습·자습**

## 완료·자습

두 모델의 검증 loss·정확도·파라미터 수·시간을 기록하고 hidden state와 cell state의 역할을 설명하세요.
자습(20분): 터미널에서 `--profile selfstudy`로 훈련하고 quick 결과와 비교합니다. seed·입력이 같은 조건인지 확인하세요.

```bash
python /workspace/04.LLM/scripts/train.py --model lstm --profile selfstudy
```

정답 확인은 [solutions/loss.py](solutions/loss.py)이며, 강사 참고 훈련 결과는 `Data/checkpoints/reference/04.LLM`에 제공합니다.


## 실험 결과 해설

**비교 순서:** 같은 과제·데이터 분할·훈련 설정을 확인하고 RNN/LSTM의 검증 loss와 token/sequence 정확도를 비교합니다. 한 token이 맞는 것과 전체 순서가 맞는 것은 다른 기준입니다. 다섯 위치 중 네 개만 맞으면 해당 예제의 token 정확도는 4/5이지만 전체 일치는 실패입니다.

RNN은 이전 상태를 다음 위치에 전달하고 LSTM은 기억을 남기고 바꾸는 연산을 추가합니다. 이름만으로 LSTM이 항상 우수하다고 결론내리지 않습니다. 훈련 전후의 같은 입력 예측, 저장 checkpoint, 재로드 결과를 근거로 설명합니다.

**확인 질문:** 서로 다른 과제의 loss 숫자만 비교해 모델 순위를 매겨도 될까요? 실제 결과를 기록하고 준비된 참고값을 자기 측정값으로 복사하지 않습니다.
